# 02 — SAR First Look: Raw Sentinel-1 GRD

**Goal:** See what raw Sentinel-1 GRD data looks like over the Marshall Fire AOI.

SAR (Synthetic Aperture Radar) looks nothing like a photograph. Instead of
recording reflected sunlight, the satellite transmits microwave pulses and
measures the strength of the returned signal (backscatter). Bright pixels
mean strong return (buildings, rough surfaces); dark pixels mean weak return
(smooth surfaces, water). The images are inherently noisy ("speckle").

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import rasterio
from rasterio.windows import from_bounds

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
# ---------------------------------------------------------------------------
# Connect to Planetary Computer STAC and search for Sentinel-1 GRD scenes
# ---------------------------------------------------------------------------

AOI_BBOX = [-105.16, 39.93, -105.07, 40.01]  # Marshall Fire area

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Pre-fire: November 2021
search_pre = catalog.search(
    collections=["sentinel-1-grd"],
    bbox=AOI_BBOX,
    datetime="2021-11-01/2021-11-30",
)
items_pre = list(search_pre.items())

# Post-fire: January 2022
search_post = catalog.search(
    collections=["sentinel-1-grd"],
    bbox=AOI_BBOX,
    datetime="2022-01-01/2022-01-31",
)
items_post = list(search_post.items())

print(f"Pre-fire scenes (Nov 2021):  {len(items_pre)}")
print(f"Post-fire scenes (Jan 2022): {len(items_post)}")

# Pick the first item from each period
item_pre = items_pre[0]
item_post = items_post[0]

print(f"\nPre-fire item:  {item_pre.id}")
print(f"  datetime: {item_pre.datetime}")
print(f"  assets:   {list(item_pre.assets.keys())}")

print(f"\nPost-fire item: {item_post.id}")
print(f"  datetime: {item_post.datetime}")
print(f"  assets:   {list(item_post.assets.keys())}")

## Read raw VV values

We read only the VV polarization band, windowed to our AOI so we don't
download the entire granule. The raw values are digital numbers (DN)
representing backscatter amplitude in linear scale.

In [ ]:
# ---------------------------------------------------------------------------
# Read the pre-fire VV band, windowed to AOI
# ---------------------------------------------------------------------------

vv_href_pre = item_pre.assets["vv"].href

with rasterio.open(vv_href_pre) as src:
    print(f"CRS:        {src.crs}")
    print(f"Full shape: {src.shape}")
    print(f"Pixel size: {src.res}")

    window = from_bounds(*AOI_BBOX, transform=src.transform)
    vv_pre_raw = src.read(1, window=window).astype(np.float32)
    transform_pre = src.window_transform(window)

print(f"\nWindowed chip shape: {vv_pre_raw.shape}")
print(f"DN range:  {np.nanmin(vv_pre_raw):.1f} -- {np.nanmax(vv_pre_raw):.1f}")
print(f"DN mean:   {np.nanmean(vv_pre_raw):.1f}")

## Visualize with percentile stretch — expect speckle noise

SAR images have multiplicative speckle noise that makes them look "grainy".
We use a 2nd / 98th percentile stretch to improve contrast.

In [ ]:
# ---------------------------------------------------------------------------
# Display the pre-fire VV chip with percentile stretch
# ---------------------------------------------------------------------------

vmin = np.nanpercentile(vv_pre_raw, 2)
vmax = np.nanpercentile(vv_pre_raw, 98)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(vv_pre_raw, cmap="gray", vmin=vmin, vmax=vmax)
ax.set_title(f"Pre-fire VV (raw DN) -- {item_pre.datetime.date()}")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.colorbar(im, ax=ax, label="DN", shrink=0.7)
plt.tight_layout()
plt.show()

## Compare pre vs post side by side

Now we read the post-fire VV the same way and display them next to each other.
Look for areas where brightness drops -- that may indicate destroyed structures
(lower backscatter from rubble vs intact buildings).

In [ ]:
# ---------------------------------------------------------------------------
# Read post-fire VV and plot pre vs post side by side
# ---------------------------------------------------------------------------

vv_href_post = item_post.assets["vv"].href

with rasterio.open(vv_href_post) as src:
    window = from_bounds(*AOI_BBOX, transform=src.transform)
    vv_post_raw = src.read(1, window=window).astype(np.float32)

print(f"Post-fire chip shape: {vv_post_raw.shape}")
print(f"Post DN range: {np.nanmin(vv_post_raw):.1f} -- {np.nanmax(vv_post_raw):.1f}")

# Use same stretch limits for both panels so they are visually comparable
combined = np.concatenate([vv_pre_raw.ravel(), vv_post_raw.ravel()])
vmin_all = np.nanpercentile(combined, 2)
vmax_all = np.nanpercentile(combined, 98)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

im1 = ax1.imshow(vv_pre_raw, cmap="gray", vmin=vmin_all, vmax=vmax_all)
ax1.set_title(f"Pre-fire VV -- {item_pre.datetime.date()}")
plt.colorbar(im1, ax=ax1, shrink=0.6, label="DN")

im2 = ax2.imshow(vv_post_raw, cmap="gray", vmin=vmin_all, vmax=vmax_all)
ax2.set_title(f"Post-fire VV -- {item_post.datetime.date()}")
plt.colorbar(im2, ax=ax2, shrink=0.6, label="DN")

for ax in (ax1, ax2):
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

plt.suptitle("Sentinel-1 VV -- Pre vs Post Marshall Fire", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()